# Bounded selected-pair Monte Carlo preparation

A four-qubit analytic example. The frozen pair acts on reference label 3 (LSB),
not the vacuum. For H=Z0, the prepared energy is -cos(2 theta), and the
requested element K[12,3] is sin(2 theta). These are independent two-level
oracles. QWC counts here are new results, separate from historical grouping.


In [ ]:
import numpy as np
from qarp import EXACT
from qarp.algorithms import SelectedPairReadout
from qarp.blocks import selected_pair_unitary
from qarp.engines import QarpEngine
from qarp.operators import QubitOperator, selected_pair_amplitudes

theta = 0.31
rotations = ((0, 1, theta),)
hamiltonian = QubitOperator("Z0")
unitary = selected_pair_unitary(4, rotations)
amplitudes = selected_pair_amplitudes(4, rotations, reference=3)
assert np.isclose(amplitudes[3], np.cos(theta))
assert np.isclose(amplitudes[12], np.sin(theta))
expected = np.zeros(16)
expected[3], expected[12] = np.cos(theta), np.sin(theta)
np.testing.assert_allclose(unitary.unitary_matrix()[:, 3], expected, atol=1e-12)


In [ ]:
exact = SelectedPairReadout(
    hamiltonian, 4, rotations, engine=QarpEngine(seed=19),
    n_shots=EXACT, real_only=True,
)
keys = [(3, 3), (3, 12), (12, 3), (3, 5)]
results = exact.estimate(keys)
np.testing.assert_allclose(
    [item.value for item in results],
    [-np.cos(2*theta), np.sin(2*theta), np.sin(2*theta), 0], atol=1e-12,
)
assert results[-1].certified_zero
assert not results[1].certified_zero
print(dict(exact.diagnostics))


In [ ]:
sampled = SelectedPairReadout(
    hamiltonian, 4, rotations, engine=QarpEngine(seed=19),
    n_shots=2000, real_only=True,
)
estimates = sampled.estimate(keys)
for estimate, target in zip(estimates[:2], [-np.cos(2*theta), np.sin(2*theta)], strict=True):
    assert abs(estimate.value - target) <= 6 * estimate.standard_error_real
before = dict(sampled.diagnostics)
sampled.estimate(keys)
assert sampled.diagnostics["submissions"] == before["submissions"]
assert sampled.diagnostics["shots"] == before["shots"]
print(dict(sampled.diagnostics))


## Optional native GPU smoke

Disabled by default. Enable only on an approved MC-capable build. A skip is not
GPU validation. The measured request path uses its own measurement engine and
never treats the requested keys as complete rows. Coefficient sampling error is
not included in MC trajectory error bars. This two-step smoke does not establish
long-time stability or scalability.


In [ ]:
import os
import qarpx as qx
from qarp.algorithms import AggregatedMonteCarlo, ResidentEvolutionConfig
from qarp.engines import CudaqEngine
from qarp.operators import ComputationalBasisSector

if os.environ.get("QARP_RUN_CUDAQ_GPU_TESTS") == "1" and qx.montecarlo_available():
    sector = ComputationalBasisSector.spin_resolved(2, 1, 1)
    mc = AggregatedMonteCarlo(
        hamiltonian, -1.0, 0.02, 0.01, 3, unitary,
        engine=CudaqEngine(), initial_walker_count=16,
        num_trajectories=1, seed=7,
        resident_evolution=ResidentEvolutionConfig(sector),
        transition_readout=sampled,
    ).build()
    print(mc.run(), dict(mc.diagnostics))
else:
    print("Native selected-pair GPU validation: UNVERIFIED (not run)")


## Frozen molecular comparisons

The separate script `scripts/validate_selected_pair_preparations.py` accepts an
external `--package-root`, one of `h2o_one_pair`, `n2_one_pair`, `n2_two_pairs`,
`lih_one_pair` via `--case`, and a new `--output` filename. It verifies the audited
fixture identity and reproduces bare/prepared sparse energies without circuit,
FCI or GPU execution. Independent molecular-integral/FCI validation and bounded
native GPU validation remain separate approval gates. Missing original fidelity
reports are not reconstructed evidence.
